# Feature Engineering and Model Preparation

This notebook prepares the cleaned BoardGameGeek dataset for machine-learning modelling.

The goal of this phase is to transform the cleaned dataset into modelling-ready inputs.

This notebook will:

* load and verify the cleaned dataset;
* define the predicion target;
* select the initial feature columns;
* exclude identifier, ranking, and post-release popularity columns;
* prepare numerical features;
* prepare text-based features from `Mechanics` and `Domains`;
* create the feature matrix `X` and target variable `y`:
* split the data into training and testing sets;
* save perpared modelling datasets;
* document the feature engineering decisions.


## Load and Verify Cleaned Dataset for Modelling

The cleaned BoardGameGeek dataset is loaded from the `data/processed` folder.

Before feature engineering begins, the dataset is checked to confirm that the correct proccessed file is being used.

This step verifies:

* the cleaned file path;
* the dataset shape;
* the expected column names;
* dupilacte rows;
* missing values in the prediction target.


In [35]:
# Import libraries used throughout this current notebook
import pandas as pd

# Import Path for creating file paths that work
# reliebly across different OS-s
from pathlib import Path


# Find the folder from where which this notebook is running
current_folder = Path.cwd()

# The notebook is inside the "notebooks" folder,
# so it parent folder is the main project folder
project_root = current_folder.parent

# Create the complete path to the cleaned dataset
cleaned_data_path = (
    project_root
    / "data"
    / "processed"
    / "bgg_cleaned.csv"
)

print("Current folder:", current_folder)
print("Project root:", project_root)
print("Cleaned dataset path:", cleaned_data_path)
print("File exists:", cleaned_data_path.exists())

Current folder: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\notebooks
Project root: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor
Cleaned dataset path: c:\Users\Zombie\Desktop\vscode-projects\board-game-rating-predictor\data\processed\bgg_cleaned.csv
File exists: True


In [36]:
# Load cleaned dataset into DataFrame for modelling preparation
df_model = pd.read_csv(cleaned_data_path)

print("Dataset loaded succeesfully.")
print("Dataset shape:", df_model.shape)


Dataset loaded succeesfully.
Dataset shape: (20342, 14)


In [37]:
# Display first 5 rows of the modelling 
# dataset (confirmation that we are good to go)
df_model.head() 

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
0,174430.0,Gloomhaven,2017.0,1.0,4.0,120.0,14.0,42055,8.79,1,3.86,68323.0,"Action Queue, Action Retrieval, Campaign / Bat...","Strategy Games, Thematic Games"
1,161936.0,Pandemic Legacy: Season 1,2015.0,2.0,4.0,60.0,13.0,41643,8.61,2,2.84,65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
2,224517.0,Brass: Birmingham,2018.0,2.0,4.0,120.0,14.0,19217,8.66,3,3.91,28785.0,"Hand Management, Income, Loans, Market, Networ...",Strategy Games
3,167791.0,Terraforming Mars,2016.0,1.0,5.0,120.0,12.0,64864,8.43,4,3.24,87099.0,"Card Drafting, Drafting, End Game Bonuses, Han...",Strategy Games
4,233078.0,Twilight Imperium: Fourth Edition,2017.0,3.0,6.0,480.0,14.0,13468,8.70,5,4.22,16831.0,"Action Drafting, Area Majority / Influence, Ar...","Strategy Games, Thematic Games"


In [38]:
# Display random 5 rows of modelling 
# dataset (as I want to be extra sure)
df_model.sample(5)

,ID,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,Rating Average,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
10164,239938.0,Kintsugi,2018.0,2.0,3.0,15.0,8.0,63,6.68,10166,1.67,254.0,"Layering, Tile Placement",Unknown
10069,163482.0,15 Dias: The Spanish Golden Age,2014.0,2.0,8.0,60.0,12.0,109,6.35,10071,2.88,262.0,"Hand Management, Variable Phase Order",Unknown
19257,221644.0,Pyramid of the Sun,2017.0,2.0,5.0,30.0,8.0,200,5.26,19260,1.50,560.0,"Hand Management, Tile Placement",Unknown
19583,5018.0,Fishin' Time,1986.0,2.0,8.0,60.0,6.0,77,4.56,19586,1.14,136.0,Roll / Spin and Move,Unknown
18447,98203.0,Redakai,2011.0,2.0,2.0,20.0,6.0,110,5.62,18450,1.67,234.0,"Hand Management, Variable Player Powers",Customizable Games


In [39]:
# Deifne the expected column names
expected_columns = [
    "ID",
    "Name",
    "Year Published",
    "Min Players",
    "Max Players",
    "Play Time",
    "Min Age",
    "Users Rated",
    "Rating Average",
    "BGG Rank",
    "Complexity Average",
    "Owned Users",
    "Mechanics",
    "Domains"
]


# Verify that the correct cleaned dataset has been loaded
print("Correct number of rows:", df_model.shape[0] == 20342)
print("Correct number of columns:", df_model.shape[1] == 14)

print(
    "Column names match:",
    df_model.columns.tolist() == expected_columns
)

print("Fully duplicated rows:", df_model.duplicated().sum())

print(
    "Missing Rating Average values:",
    df_model["Rating Average"].isnull().sum()
) 

Correct number of rows: True
Correct number of columns: True
Column names match: True
Fully duplicated rows: 0
Missing Rating Average values: 0


### Cleaned Dataset Loading Resutls

The cleaned dataset was loaded successfully from `data/processed/bgg_cleaned.csv`.

The dataset contain 20,342 rows and 14 columns, matching the expected processd dataset created during the data cleaning phase.

The expected column names are present, there are no fully duplicated rows, and the prediction target `Rating Average` contains no missing values.

This confirms that the feature engineering notebook is starting from the correct cleaned dataset.


## Define Target and Feature Columns

The prediction target an possible feature columns are defined before modelling preparation begins.

The target is the value the machine-learning model will try to predict.

The feature columns are the information the model may use to make that prediction.

For this proejct, the prediction target is `Rating Average`.

The initial feature plan focuses on board game design information such as publicaton year, player counts, play time, recommended age, complexity, mehanics, and domains.


In [40]:
# Define prediction target
target_column = "Rating Average"

print("Target column:", target_column)
print("Target data type:", df_model[target_column].dtype)
print("Missing target values:", df_model[target_column].isnull().sum())

Target column: Rating Average
Target data type: float64
Missing target values: 0


In [41]:
# Define initial numerical feature candidates
numeric_feature_candidates = [
    "Year Published",
    "Min Players",
    "Max Players",
    "Play Time",
    "Min Age",
    "Complexity Average"
]

# Define initial text-based feature candidates
text_feature_candidates = [
    "Mechanics",
    "Domains"
]

# Define columns that need to be reviewed for excluson
columns_to_review_for_exclusion = [
    "ID",
    "Name",
    "BGG Rank",
    "Users Rated",
    "Owned Users"
]

print("Numerical feature candidates:", numeric_feature_candidates)
print("Text feature candidates:", text_feature_candidates)
print("Columns to review for exclusion:", columns_to_review_for_exclusion)

Numerical feature candidates: ['Year Published', 'Min Players', 'Max Players', 'Play Time', 'Min Age', 'Complexity Average']
Text feature candidates: ['Mechanics', 'Domains']
Columns to review for exclusion: ['ID', 'Name', 'BGG Rank', 'Users Rated', 'Owned Users']


In [42]:
# Create summary table of the target and feature groups
feature_definition_summary = pd.DataFrame(
    [
        {
            "Column": target_column,
            "Role": "Target",
            "Reason": "Value the model will predict"
        },
        *[
            {
                "Column": column,
                "Role": "Numerical feature candidate",
                "Reason": "Board game design or description information"
            }
            for column in numeric_feature_candidates
        ],
        *[
            {
                "Column": column,
                "Role": "Text feature candidate",
                "Reason": "Multi-value board game category information"
            }
            for column in text_feature_candidates
        ],
        *[
            {
                "Column": column,
                "Role": "Review for exclusion",
                "Reason": "Identifier, name, ranjing, or post-release popularity information"
            }
            for column in columns_to_review_for_exclusion
        ]
    ]
)

feature_definition_summary

,Column,Role,Reason
0,Rating Average,Target,Value the model will predict
1,Year Published,Numerical feature candidate,Board game design or description information
2,Min Players,Numerical feature candidate,Board game design or description information
3,Max Players,Numerical feature candidate,Board game design or description information
4,Play Time,Numerical feature candidate,Board game design or description information
5,Min Age,Numerical feature candidate,Board game design or description information
6,Complexity Average,Numerical feature candidate,Board game design or description information
7,Mechanics,Text feature candidate,Multi-value board game category information
8,Domains,Text feature candidate,Multi-value board game category information
9,ID,Review for exclusion,"Identifier, name, ranjing, or post-release pop..."


In [43]:
# Combine all defined columns into the one list
defined_columns = (
    [target_column]
    + numeric_feature_candidates
    + text_feature_candidates
    + columns_to_review_for_exclusion
)

# Check that for any columns that were defined but are not present in the dataset
missing_defined_columns = [
    column
    for column in defined_columns
    if column not in df_model.columns
]

# Check that all dataset columns have been assigned to the one of the groups
unassigned_columns = [
    column
    for column in df_model.columns
    if column not in defined_columns
]

print("Missing defined columns:", missing_defined_columns)
print("Unassigned dataset columns:", unassigned_columns)
print("Total defined columns:", len(defined_columns))
print("Total dataset columns:", len(df_model.columns))

Missing defined columns: []
Unassigned dataset columns: []
Total defined columns: 14
Total dataset columns: 14


### Target and Feature Column Definition Result

The prediction target was defind as `Rating Average`.

The initial numerical feature candidates are:

* `Year Published`
* `Min Players`
* `Max Players`
* `Play Tine`
* `Min Age`
* `Complexity Average`

The initial text-based feature candidates are:

* `Mechanics`
* `Domains`

The following columns were idetified for exclusion review:

* `ID`
* `Name`
* `BGG Rank`
* `Users Rated`
* `Owned Users`

All 14 columns in the cleaned dataset have been assigned to the target, feature candidate, or exclusion-review group.

Not data was changed during this step.


## Exclude Leakage and Identifier Columns

Some columns should not be used as model features.

Idebtifier columns, ranking columns, and post-release popularity columns may either provide no usefoul learning signal or give the model information that would not be available before a game receives public ratings.

This step separates useable feature columns from columns that should be excluded from the initial modelling dataset.


In [44]:
# Define columns that will be excluded from the initial model features
excluded_columns = {
    "ID": "Identifier column; does not descirbe board game design.",
    "Name": "Game title; may cause the model to memorise specific games rather than learn general patterns.",
    "BGG Rank": "Ranking infromation based on BoardGameGeek performance; likely target leakage.",
    "Users Rated": "Post-release popularity information; not suitable for initial pre-release-style prediction.",
    "Owned Users": "Post-release ownership/popularity information; not suitable for initial pre-release-style prediction."
}

excluded_columns

{'ID': 'Identifier column; does not descirbe board game design.',
 'Name': 'Game title; may cause the model to memorise specific games rather than learn general patterns.',
 'BGG Rank': 'Ranking infromation based on BoardGameGeek performance; likely target leakage.',
 'Users Rated': 'Post-release popularity information; not suitable for initial pre-release-style prediction.',
 'Owned Users': 'Post-release ownership/popularity information; not suitable for initial pre-release-style prediction.'}

In [45]:
# Create a readable table explaining why each 
# of the column is excluded
exclusion_summary = pd.DataFrame(
    [
        {
            "Column": column,
            "Reason for Exculsion": reason
        }
        for column, reason in excluded_columns.items()
    ]
)

exclusion_summary

,Column,Reason for Exculsion
0,ID,Identifier column; does not descirbe board gam...
1,Name,Game title; may cause the model to memorise sp...
2,BGG Rank,Ranking infromation based on BoardGameGeek per...
3,Users Rated,Post-release popularity information; not suita...
4,Owned Users,Post-release ownership/popularity information;...


In [46]:
# Define selectd feature columns for the initial model
selected_numeric_features = numeric_feature_candidates.copy()
selected_text_features = text_feature_candidates.copy()

selected_feature_columns = (
    selected_numeric_features
    + selected_text_features
)

print("Selected numerical features:", selected_numeric_features)
print("Selected text features:", selected_text_features)
print("All of selected feature columns:", selected_feature_columns)

Selected numerical features: ['Year Published', 'Min Players', 'Max Players', 'Play Time', 'Min Age', 'Complexity Average']
Selected text features: ['Mechanics', 'Domains']
All of selected feature columns: ['Year Published', 'Min Players', 'Max Players', 'Play Time', 'Min Age', 'Complexity Average', 'Mechanics', 'Domains']


In [47]:
# Check that excluded columns are not present in 
# the selected feature columns -fail safes
excluded_columns_in_features = [
    column
    for column in excluded_columns
    if column in selected_feature_columns
]

# Check that the target column is not accidentally included as a feature
target_in_features = target_column in selected_feature_columns

print("Exluded columns accidentally included as features:", excluded_columns_in_features)
print("Target column accidentally included as feature:", target_in_features)
print("Number of selected feature columns:", len(selected_feature_columns))

Exluded columns accidentally included as features: []
Target column accidentally included as feature: False
Number of selected feature columns: 8


In [48]:
# Create modelling column summary
modelling_column_summary = pd.DataFrame(
    [
        {
            "Column": target_column,
            "Modelling Role": "Target"
        },
        *[
            {
                "Column": column,
                "Modelling Role": "Selected numerical feature"
            }
            for column in selected_numeric_features
        ],
        *[
            {
                "Column": column,
                "Modelling Role": "Selected text feature"
            }
            for column in selected_text_features
        ],
        *[
            {
                "Column": column,
                "Modelling Role": "Excluded"
            }
            for column in excluded_columns
        ]
    ]
)

modelling_column_summary

,Column,Modelling Role
0,Rating Average,Target
1,Year Published,Selected numerical feature
2,Min Players,Selected numerical feature
3,Max Players,Selected numerical feature
4,Play Time,Selected numerical feature
5,Min Age,Selected numerical feature
6,Complexity Average,Selected numerical feature
7,Mechanics,Selected text feature
8,Domains,Selected text feature
9,ID,Excluded


### Leakage and Identifier Column Exclusion Result

The initial model feature set excludes `ID`, `Name`, `BGG Rank`, `Users Rated`, and `Owned Users`.

`ID` is excluded because it is only an identifier and does not help us as it does not describe board game design.

`Name` is excluded because game titles may encourage memorisation (relation to brand, movie, TV show, book) rather than general learning.

`BGG Rank` is excluded because it is ranking information closely related to rating performance and may introduce target leakage.

`Users Rated` and `Owned Users` are excluded because they describe post-release popularity. These values would not be suitable for an initial model focused on predicting ratings from board game design information.

The selected initial features are six numericla columns and two text-based columns.

Not a single excluded columns was accidentally included in the selected feature list, and the target column was not included as a feature.
